<a href="https://colab.research.google.com/github/Yash-Agarwal-4a5h/cei-assignments-yash-agarwal-jecrc-college/blob/main/Week8_yash_agarwal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Smart Agent Pipeline & Quiz Evaluator

**Part 1** — Building a single-agent pipeline with conditional routing, a math evaluation tool, and a keyword extraction tool.

**Part 2** — Using the pipeline to auto-grade Week 8 quiz answers based on reference keywords.

## Part 1 — Agent Pipeline Construction

In [ ]:
import re
import json
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
agent_logger = logging.getLogger("smart_agent")


In [ ]:
class MathEvaluatorTool:
    """Tool for evaluating math expressions present in the user query."""
    name = "math_tool"

    def execute(self, query: str) -> dict:
        match = re.search(r"[\d\.\s\+\-\*\/\(\)]+", query)
        if not match:
            raise ValueError("No valid mathematical expression found")
        expression = match.group().strip()

        if not re.fullmatch(r"[\d\.\s\+\-\*\/\(\)]+", expression):
            raise ValueError("Expression is unsafe for evaluation")
        result = eval(expression, {"__builtins__": {}})
        return {"tool": self.name, "expression": expression, "result": result}


IGNORE_WORDS = {
    "the","a","an","is","are","was","were","in","on","of","to","and","or","for",
    "with","that","this","it","as","by","be","from","at","which","how","what",
    "why","when","can","could","would","should","if","then","than","its","their",
    "these","those","also","such","into","over","between","each","other","more",
    "so","do","does","did","not","no","but","has","have","had","will","may",
    "used","use","using","provide","give","example","explain","describe","define",
}

class SalientWordExtractor:
    """Extracts important keywords from text by filtering out stop words."""
    name = "keyword_extractor"

    def execute(self, query: str) -> dict:
        text = query.lower()
        text = re.sub(r"[^a-z0-9\s\-]", " ", text)
        words = [w.strip("-") for w in text.split()]
        words = [w for w in words if w and w not in IGNORE_WORDS and len(w) > 2]

        word_counts = {}
        for w in words:
            word_counts[w] = word_counts.get(w, 0) + 1
        top_keywords = sorted(set(words), key=lambda w: -word_counts[w])
        return {"tool": self.name, "keywords": top_keywords, "count": len(top_keywords)}


In [ ]:
class SmartAgent:
    def __init__(self):
        self.math_tool = MathEvaluatorTool()
        self.extractor_tool = SalientWordExtractor()
        self.retry_limit = 3

    def determine_route(self, query: str) -> str:
        q = query.lower()
        if "calculate" in q or re.search(r"\d+\s*[\+\-\*\/]\s*\d+", q):
            return "process_math"
        if "keyword" in q:
            return "process_keyword"
        return "process_general"

    def _execute_with_retry(self, func, *args):
        latest_error = None
        for attempt in range(1, self.retry_limit + 1):
            try:
                return func(*args)
            except Exception as ex:
                latest_error = ex
                agent_logger.info(f"Attempt {attempt} failed: {ex}")
                time.sleep(0.5)
        return {"error": str(latest_error)}

    def process_math(self, query, context):
        result = self._execute_with_retry(self.math_tool.execute, query)
        context["result"] = result
        context["path"].append("math_evaluation")
        return context

    def process_keyword(self, query, context):
        result = self._execute_with_retry(self.extractor_tool.execute, query)
        context["result"] = result
        context["path"].append("keyword_extraction")
        return context

    def process_general(self, query, context):
        context["result"] = {"tool": "general_fallback", "response": f"Direct response provided for: {query}"}
        context["path"].append("general_handling")
        return context

    def process_query(self, query: str) -> dict:
        context = {"query": query, "path": ["router"], "result": None}
        node = self.determine_route(query)
        handler = getattr(self, node)
        context = handler(query, context)
        return context

my_agent = SmartAgent()


In [ ]:
test_prompts = [
    "calculate 12 + 8 * 2",
    "extract keywords from this sentence about agents",
    "tell me about the weather"
]

for prompt in test_prompts:
    print(f"User: {prompt}")
    print(f"Agent Output: {my_agent.process_query(prompt)}\n")


User: calculate 12 + 8 * 2
Agent Output: {'query': 'calculate 12 + 8 * 2', 'path': ['router', 'math_evaluation'], 'result': {'tool': 'math_tool', 'expression': '12 + 8 * 2', 'result': 28}}

User: extract keywords from this sentence about agents
Agent Output: {'query': 'extract keywords from this sentence about agents', 'path': ['router', 'keyword_extraction'], 'result': {'tool': 'keyword_extractor', 'keywords': ['extract', 'about', 'keywords', 'agents', 'sentence'], 'count': 5}}

User: tell me about the weather
Agent Output: {'query': 'tell me about the weather', 'path': ['router', 'general_handling'], 'result': {'tool': 'general_fallback', 'response': 'Direct response provided for: tell me about the weather'}}



## Part 2 — Auto-Grading System

In [ ]:
QUIZ_DATA = [
    {
        "id": 1,
        "question": "What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.",
        "student_answer": "Conditional routing is the process of directing a query to different tools or actions based on its intent. It helps the agent choose the most appropriate response method. For example, if a query contains the word calculate, it is sent to the Calculator Tool. If it contains keywords, it is sent to the Keyword Extraction Tool. All other queries can be handled by a General Response module.",
        "reference_keywords": ["conditional routing", "intent", "calculator tool", "keyword extraction tool", "general response", "rule-based"]
    },
    {
        "id": 2,
        "question": "Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?",
        "student_answer": "A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges that define how data moves through the system. It can support branching, looping, and decision-making. This makes it more flexible and intelligent. In contrast, a linear pipeline follows a fixed sequence of steps from start to finish without remembering previous states or changing its path.",
        "reference_keywords": ["state", "nodes", "edges", "branching", "looping", "decision-making", "linear", "fixed sequence"]
    },
    {
        "id": 3,
        "question": "Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?",
        "student_answer": "Sequential tool calls are executed one after another, where each step depends on the result of the previous step. Parallel tool calls execute multiple independent tasks at the same time. Sequential execution is preferred when tasks have dependencies. Parallel execution is useful when tasks are unrelated and can be completed simultaneously. Using parallel calls can significantly reduce response time and improve efficiency.",
        "reference_keywords": ["sequential", "parallel", "dependencies", "independent tasks", "response time", "efficiency"]
    },
    {
        "id": 4,
        "question": "What are JSON schema tools? How do they help in structuring tool inputs and outputs?",
        "student_answer": "JSON schema tools define a standard structure for data exchanged between agents and tools. They specify required fields, data types, and expected formats. This ensures that inputs are valid before processing begins. It also makes outputs consistent and easier to interpret. Using JSON schemas reduces errors and improves communication between different components of an agent system.",
        "reference_keywords": ["json schema", "standard structure", "required fields", "data types", "validation", "consistent output"]
    },
    {
        "id": 5,
        "question": "Explain how a single-agent system can simulate multi-agent behavior internally.",
        "student_answer": "A single-agent system can simulate multiple agents by dividing its work into different roles. It may first analyze the query, then decide which tool to use, and finally generate a response. Each role behaves like a separate agent even though they are all executed by one system. This approach reduces complexity while maintaining flexibility. As a result, a single agent can perform tasks similar to a multi-agent architecture.",
        "reference_keywords": ["roles", "divide", "analyze query", "decide tool", "generate response", "single system", "multi-agent"]
    },
    {
        "id": 6,
        "question": "Describe the role of nodes and edges in an agent workflow. Give an example of each.",
        "student_answer": "Nodes are the individual tasks or actions performed in an agent workflow. They can represent operations such as processing a query, calling a tool, or generating a response. Edges are the connections between nodes that determine how information flows. For example, a Calculator Tool can be a node, while the path connecting query analysis to the calculator is an edge. Together, nodes and edges define the complete workflow.",
        "reference_keywords": ["nodes", "tasks", "actions", "edges", "connections", "information flow", "example"]
    },
    {
        "id": 7,
        "question": "Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?",
        "student_answer": "Task completion rate measures the percentage of tasks successfully completed by an agent. Cost metrics represent the resources consumed, such as API calls, processing time, or monetary expenses. These metrics can be measured by tracking agent performance over multiple tasks. To optimize them, developers can improve routing accuracy, reduce unnecessary tool calls, and use efficient algorithms.",
        "reference_keywords": ["completion rate", "cost metrics", "resources", "api calls", "processing time", "routing accuracy", "optimize"]
    },
    {
        "id": 8,
        "question": "Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.",
        "student_answer": "Cycles or loops allow an agent to repeat a process until a desired result is achieved. They are useful when tasks may fail or require multiple attempts. For example, if an API request fails due to a temporary network issue, the agent can retry the request instead of immediately returning an error. This improves reliability and robustness. Without loops, the workflow would stop after a single failure.",
        "reference_keywords": ["cycles", "loops", "repeat", "retry", "api request", "failure", "reliability"]
    },
    {
        "id": 9,
        "question": "What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?",
        "student_answer": "Trajectory evaluation examines the entire sequence of actions taken by an agent to solve a task. It focuses on the decisions, tool calls, and intermediate steps used during execution. This helps identify mistakes that may not be visible in the final answer. It is useful for debugging, optimization, and improving agent performance. By evaluating the full trajectory, developers gain a deeper understanding of agent behavior.",
        "reference_keywords": ["trajectory", "sequence of actions", "decisions", "tool calls", "intermediate steps", "debugging", "optimization"]
    },
    {
        "id": 10,
        "question": "How would you implement error handling in a tool-using agent? Provide at least two strategies.",
        "student_answer": "Error handling helps an agent continue operating even when problems occur. One strategy is using try-except blocks to catch exceptions and return meaningful error messages. Another strategy is implementing retry mechanisms that automatically repeat failed operations. Logging errors is also useful for debugging and monitoring. These techniques improve reliability and help maintain a better user experience.",
        "reference_keywords": ["try-except", "exceptions", "retry mechanism", "logging", "debugging", "reliability"]
    }
]
print(f"Loaded {len(QUIZ_DATA)} questions successfully.")


Loaded 10 questions successfully.


In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def evaluate_response(item: dict, student_input: str, extractor_tool: SalientWordExtractor) -> dict:
    """Grade the given answer based on reference keywords."""
    extracted_words = extractor_tool.execute(student_input)["keywords"]
    normalized_input = clean_text(student_input)

    matched_words, missed_words = [], []
    for reference in item["reference_keywords"]:
        ref_norm = clean_text(reference)

        if ref_norm in normalized_input or all(w in normalized_input for w in ref_norm.split()):
            matched_words.append(reference)
        else:
            missed_words.append(reference)

    score_ratio = len(matched_words) / len(item["reference_keywords"])
    return {
        "question_id": item["id"],
        "extracted_keywords": extracted_words[:10],
        "matched_words": matched_words,
        "missed_words": missed_words,
        "final_score": round(score_ratio, 2),
    }


In [ ]:
def execute_quiz_grader(quiz_items: list, extractor_tool: SalientWordExtractor) -> list:
    """Iterate over the quiz questions and grade student inputs automatically."""
    grading_results = []
    accumulated_score = 0

    for item in quiz_items:
        q_id = item["id"]
        q_text = item["question"]
        print(f"\nQuestion {q_id}: {q_text}")

        # Automatically use the pre-filled student answer for a perfect score!
        student_answer = item.get("student_answer", "")
        print(f"Answer Provided: {student_answer}")

        evaluation = evaluate_response(item, student_answer, extractor_tool)
        grading_results.append(evaluation)
        accumulated_score += evaluation["final_score"]

        percentage = evaluation["final_score"] * 100
        found = evaluation["matched_words"]
        missing = evaluation["missed_words"]
        print(f"Result -> Score: {percentage:.0f}% | Matched: {found} | Missed: {missing}")

    average_score = accumulated_score / len(quiz_items)
    print(f"\nFINAL SCORE: {average_score*100:.1f}% ({accumulated_score:.2f} out of {len(quiz_items)})")
    return grading_results

final_results = execute_quiz_grader(QUIZ_DATA, my_agent.extractor_tool)



Question 1: What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.
Answer Provided: Conditional routing is the process of directing a query to different tools or actions based on its intent. It helps the agent choose the most appropriate response method. For example, if a query contains the word calculate, it is sent to the Calculator Tool. If it contains keywords, it is sent to the Keyword Extraction Tool. All other queries can be handled by a General Response module.
Result -> Score: 83% | Matched: ['conditional routing', 'intent', 'calculator tool', 'keyword extraction tool', 'general response'] | Missed: ['rule-based']

Question 2: Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?
Answer Provided: A stateful directed graph is a workflow where each step can store and use information from previous steps. The workflow is made up of nodes and edges th